## Gene-level TSS

In [1]:
library(AnnotationDbi)
library(org.Hs.eg.db)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(dplyr)
library(rtracklayer)

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Loading required package: IRanges

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The f

In [2]:
# Load gene symbols
candidate_genes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/mega_upregulated.tsv", header = FALSE)[[1]]

In [3]:
# Map gene symbols to Entrez IDs
entrez_ids <- mapIds(org.Hs.eg.db,
                     keys = candidate_genes,
                     column = "ENTREZID",
                     keytype = "SYMBOL",
                     multiVals = "first")

'select()' returned 1:1 mapping between keys and columns



In [4]:
# Get TSS from TxDb
tss_coords <- genes(TxDb.Hsapiens.UCSC.hg38.knownGene)

  2135 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [5]:
# Filter for my genes only here
tss_subset <- tss_coords[tss_coords$gene_id %in% entrez_ids]

In [6]:
# Extract TSS as GRanges object
tss_ranges <- promoters(tss_subset, upstream = 0, downstream = 1)

In [7]:
tss_ranges

GRanges object with 190 ranges and 1 metadata column:
            seqnames    ranges strand |     gene_id
               <Rle> <IRanges>  <Rle> | <character>
       1001    chr16  68644993      + |        1001
  100131378    chr11  33722448      - |   100131378
  100652740    chr16  31201885      + |   100652740
      10112     chr5 138178719      + |       10112
      10262     chr1 149927803      - |       10262
        ...      ...       ...    ... .         ...
       9601     chr7 149028662      - |        9601
       9672     chr1  30908758      - |        9672
        976    chr19  14380501      + |         976
        991     chr1  43358981      + |         991
       9918    chr12   6493356      + |        9918
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

In [8]:
# add name column
mcols(tss_ranges)$name <- mcols(tss_ranges)$gene_id

#Add score column
mcols(tss_ranges)$score <- 0

# Export as BED
export(tss_ranges, "mega_upregulated_TSS.bed", format = "BED")


In [9]:
# promoter - motif or TF binding
tss_core_promoter <- promoters(tss_subset, upstream = 2000, downstream = 500)
# add name column
mcols(tss_core_promoter)$name <- mcols(tss_core_promoter)$gene_id

#Add score column
mcols(tss_core_promoter)$score <- 0

# Export as BED
export(tss_core_promoter, "mega_upregulated_corePromoter.bed", format = "BED")

In [10]:
# to do - for down regulated

In [11]:
# Load gene symbols
candidate_genes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/mega_downregulated.tsv", header = FALSE)[[1]]

In [12]:
# Map gene symbols to Entrez IDs
entrez_ids <- mapIds(org.Hs.eg.db,
                     keys = candidate_genes,
                     column = "ENTREZID",
                     keytype = "SYMBOL",
                     multiVals = "first")

'select()' returned 1:1 mapping between keys and columns



In [13]:
# Get TSS from TxDb
tss_coords <- genes(TxDb.Hsapiens.UCSC.hg38.knownGene)

  2135 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [14]:
# Filter for my genes only here
tss_subset <- tss_coords[tss_coords$gene_id %in% entrez_ids]

In [15]:
# Extract TSS as GRanges object
tss_ranges <- promoters(tss_subset, upstream = 0, downstream = 1)

In [16]:
# add name column
mcols(tss_ranges)$name <- mcols(tss_ranges)$gene_id

#Add score column
mcols(tss_ranges)$score <- 0

# Export as BED
export(tss_ranges, "mega_downregulated_TSS.bed", format = "BED")

---

## Transcript -level TSS

##### first up regulated

In [17]:
mygenes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/mega_upregulated.tsv", header = FALSE)[[1]]

In [18]:
library(org.Hs.eg.db)

# Map gene symbols to Entrez IDs
mapped <- mapIds(org.Hs.eg.db,
                 keys = mygenes,
                 column = "ENTREZID",
                 keytype = "SYMBOL",
                 multiVals = "first")

# Remove NAs
entrez_ids <- na.omit(mapped)

'select()' returned 1:1 mapping between keys and columns



In [19]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
mygenes.transcripts <- subset(
  transcripts(txdb, columns = c("tx_id", "tx_name", "gene_id")),
  gene_id %in% entrez_ids
)

In [20]:
mygenes.tss <- resize(mygenes.transcripts, width=1, fix="start")

In [21]:
as.data.frame(mygenes.tss) # TSS position per transcript only

seqnames,start,end,width,strand,tx_id,tx_name,gene_id
<fct>,<int>,<int>,<int>,<fct>,<int>,<chr>,<list>
chr1,11934205,11934205,1,+,955,ENST00000485046.5,5351
chr1,11934694,11934694,1,+,956,ENST00000449038.5,5351
chr1,11934717,11934717,1,+,957,ENST00000196061.5,5351
chr1,11934734,11934734,1,+,958,ENST00000358133.5,5351
chr1,11934743,11934743,1,+,959,ENST00000429000.6,5351
chr1,11954301,11954301,1,+,960,ENST00000465920.1,5351
chr1,11964688,11964688,1,+,961,ENST00000491536.5,5351
chr1,11964702,11964702,1,+,962,ENST00000470133.1,5351
chr1,11971543,11971543,1,+,963,ENST00000481933.1,5351


In [22]:
# Converting to dataframe
tss_df <- as.data.frame(mygenes.tss)

# Creating a .bed like table
bed_tss_df <- data.frame(
  chrom = tss_df$seqnames,
  chromStart = tss_df$start - 1,  # BED is 0-based
  chromEnd = tss_df$start,        # 1 bp region
  name = tss_df$tx_name,          
  score = 0,                      # just a standard
  strand = tss_df$strand
)

# Export to BED
write.table(bed_tss_df, file = "upregulated_transcript_level_TSS.bed",
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)

In [23]:
tss_df

seqnames,start,end,width,strand,tx_id,tx_name,gene_id
<fct>,<int>,<int>,<int>,<fct>,<int>,<chr>,<list>
chr1,11934205,11934205,1,+,955,ENST00000485046.5,5351
chr1,11934694,11934694,1,+,956,ENST00000449038.5,5351
chr1,11934717,11934717,1,+,957,ENST00000196061.5,5351
chr1,11934734,11934734,1,+,958,ENST00000358133.5,5351
chr1,11934743,11934743,1,+,959,ENST00000429000.6,5351
chr1,11954301,11954301,1,+,960,ENST00000465920.1,5351
chr1,11964688,11964688,1,+,961,ENST00000491536.5,5351
chr1,11964702,11964702,1,+,962,ENST00000470133.1,5351
chr1,11971543,11971543,1,+,963,ENST00000481933.1,5351


In [24]:
bed_tss_df

chrom,chromStart,chromEnd,name,score,strand
<fct>,<dbl>,<int>,<chr>,<dbl>,<fct>
chr1,11934204,11934205,ENST00000485046.5,0,+
chr1,11934693,11934694,ENST00000449038.5,0,+
chr1,11934716,11934717,ENST00000196061.5,0,+
chr1,11934733,11934734,ENST00000358133.5,0,+
chr1,11934742,11934743,ENST00000429000.6,0,+
chr1,11954300,11954301,ENST00000465920.1,0,+
chr1,11964687,11964688,ENST00000491536.5,0,+
chr1,11964701,11964702,ENST00000470133.1,0,+
chr1,11971542,11971543,ENST00000481933.1,0,+


##### now downregulated

In [25]:
mygenes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/mega_downregulated.tsv", header = FALSE)[[1]]

In [26]:
library(org.Hs.eg.db)

# Map gene symbols to Entrez IDs
mapped <- mapIds(org.Hs.eg.db,
                 keys = mygenes,
                 column = "ENTREZID",
                 keytype = "SYMBOL",
                 multiVals = "first")

# Remove NAs
entrez_ids <- na.omit(mapped)

'select()' returned 1:1 mapping between keys and columns



In [27]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
mygenes.transcripts <- subset(
  transcripts(txdb, columns = c("tx_id", "tx_name", "gene_id")),
  gene_id %in% entrez_ids
)

In [28]:
mygenes.tss <- resize(mygenes.transcripts, width=1, fix="start")

In [29]:
as.data.frame(mygenes.tss) # TSS position per transcript only

seqnames,start,end,width,strand,tx_id,tx_name,gene_id
<fct>,<int>,<int>,<int>,<fct>,<int>,<chr>,<list>
chr1,960584,960584,1,+,130,ENST00000338591.8,339451
chr1,961449,961449,1,+,131,ENST00000463212.1,339451
chr1,962727,962727,1,+,132,ENST00000466300.1,339451
chr1,963552,963552,1,+,133,ENST00000481067.1,339451
chr1,1615454,1615454,1,+,245,ENST00000479659.5,142678
chr1,1615496,1615496,1,+,246,ENST00000489635.5,142678
chr1,1615500,1615500,1,+,247,ENST00000355826.10,142678
chr1,1615500,1615500,1,+,248,ENST00000505820.7,142678
chr1,1615500,1615500,1,+,249,ENST00000518681.6,142678


In [30]:
# Converting to dataframe
tss_df <- as.data.frame(mygenes.tss)

# Creating a .bed like table
bed_tss_df <- data.frame(
  chrom = tss_df$seqnames,
  chromStart = tss_df$start - 1,  # BED is 0-based
  chromEnd = tss_df$start,        # 1 bp region
  name = tss_df$tx_name,          
  score = 0,                      # just a standard
  strand = tss_df$strand
)

# Export to BED
write.table(bed_tss_df, file = "downregulated_transcript_level_TSS.bed",
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)

In [31]:
tss_df

seqnames,start,end,width,strand,tx_id,tx_name,gene_id
<fct>,<int>,<int>,<int>,<fct>,<int>,<chr>,<list>
chr1,960584,960584,1,+,130,ENST00000338591.8,339451
chr1,961449,961449,1,+,131,ENST00000463212.1,339451
chr1,962727,962727,1,+,132,ENST00000466300.1,339451
chr1,963552,963552,1,+,133,ENST00000481067.1,339451
chr1,1615454,1615454,1,+,245,ENST00000479659.5,142678
chr1,1615496,1615496,1,+,246,ENST00000489635.5,142678
chr1,1615500,1615500,1,+,247,ENST00000355826.10,142678
chr1,1615500,1615500,1,+,248,ENST00000505820.7,142678
chr1,1615500,1615500,1,+,249,ENST00000518681.6,142678


In [32]:
bed_tss_df

chrom,chromStart,chromEnd,name,score,strand
<fct>,<dbl>,<int>,<chr>,<dbl>,<fct>
chr1,960583,960584,ENST00000338591.8,0,+
chr1,961448,961449,ENST00000463212.1,0,+
chr1,962726,962727,ENST00000466300.1,0,+
chr1,963551,963552,ENST00000481067.1,0,+
chr1,1615453,1615454,ENST00000479659.5,0,+
chr1,1615495,1615496,ENST00000489635.5,0,+
chr1,1615499,1615500,ENST00000355826.10,0,+
chr1,1615499,1615500,ENST00000505820.7,0,+
chr1,1615499,1615500,ENST00000518681.6,0,+
